In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# models generation directory: LLM4DC/CoT.response
# predicted dataset by model:  LLM4DC/CoT.response/{model}/datasets_llm
# predicted workflow by model:  LLM4DC/CoT.response/{model}/recipes_llm
# predicted operations by model:  LLM4DC/CoT.response/{model}/operation
# logging:  LLM4DC/CoT.response/logging

# all the sample tables are under: LLM4DC/datasets
# query: 1-30 [menu]: LLM4DC/datasets/menu_datasets
# clean table (ground truth) LLM4DC/datasets/menu_datasets/clean_tables/
# clean workflow (silver ground truth):  LLM4DC/datasets/menu_datasets/workflows

# query: 31-61 [chicago]: LLM4DC/datasets/CFI_datasets
# clean table (ground truth): LLM4DC/datasets/CFI_datasets/cleaned_tables/
# clean workflow (silver ground truth):  LLM4DC/datasets/CFI_datasets/workflows

# query: 62-91 [ppp]:LLM4DC/datasets/ppp_datasets
# clean table (ground truth): LLM4DC/datasets/ppp_datasets/cleaned_tables/
# clean workflow (silver ground truth):  LLM4DC/datasets/ppp_datasets/workflows

# query: 92-110 [dish]: LLM4DC/datasets/dish_datasets
# clean table (ground truth): LLM4DC/datasets/dish_datasets/cleaned_tables/
# clean workflow (silver ground truth):  LLM4DC/datasets/dish_datasets/workflows

# query: 111-126 [flights]: LLM4DC/datasets/flights
# clean table (ground truth): LLM4DC/datasets/flights/cleaned_tables/flights_data_p{query_id}.csv
# clean workflow (silver ground truth):  LLM4DC/datasets/flights/workflows/flights_p{query_id}.json

# query: 127- 154 [hospital]: LLM4DC/datasets/hospital
# clean table (ground truth): LLM4DC/datasets/hospital/clean_tables/
# clean workflow (silver ground truth):  LLM4DC/datasets/hospital/workflows

In [2]:
import re

In [3]:
import json

In [4]:
import ast
# LLM-based history update solution
import importlib.util
import inspect
from typing import List
import requests
import json
import re
import difflib
from collections import Counter
# from spellchecker import SpellChecker
from datetime import datetime
import pandas as pd
import ast
import random
import logging 
# from history_update_problem.call_or import export_rows
from call_or import *
from evaluation import *

In [5]:
models = ['llama3.1',  'mistral', 'gemma2','gemma2base']
base_models = ['llama3.1',  'mistral', 'gemma2','gemma2base'] # locate in ablation folder

# Worflow eval

In [8]:
def eval_workflows(pp_id, gt_wf_fp, pred_wf_fp):
    # print(gt_wf_fp)
    gt_ops_list = parse_recipe(pp_id, recipe=gt_wf_fp)
    pred_ops_list = parse_recipe(pp_id, recipe=pred_wf_fp)
    return {'pp_id': pp_id, 'gt_ops': gt_ops_list[pp_id], 'pred_ops': pred_ops_list[pp_id]}
    # print(gt_ops_list)
    # print(pred_ops_list)

answer_gt_path = '/projects/bces/lanl2/LLM4DC/evaluation/answer_1-154_gt.json'
answer_preds_llama = '/projects/bces/lanl2/LLM4DC/evaluation/answer_1-154_llama3.1.json'


# eval_table_results = eval_answers(answer_gt_path, answer_preds_llama)
# model = models[0]
wf_gt_folder = '/projects/bces/lanl2/LLM4DC/datasets'

query_contents = pd.read_csv('/projects/bces/lanl2/LLM4DC/purposes/all_purposes.csv')
    
ops_result = {}
for model in models[:]:
    ops_list = []

    wf_pred_folder = f'/projects/bces/lanl2/LLM4DC/CoT.response/{model}/recipes_llm'
    for query_id in range(155):
        row = query_contents[query_contents['ID'] == query_id]
        if len(row) == 0:
            continue
        # if model == 'llama3.1':
        if query_id >126:
            #TODO: what's the point to have the target_path here? 
            target_path = f'{wf_gt_folder}/hospital/clean_tables/hos_pp{query_id}.csv'
            wf_gt_fp = f"{wf_gt_folder}/hospital/workflows/hos_p{query_id}.json"
            wf_pred_fp = f"{wf_pred_folder}/{model}_hos_test_p{query_id}.json"
        elif query_id >= 111 and query_id <=126:
            target_path = f'{wf_gt_folder}/flights/cleaned_tables/flights_data_p{query_id}.csv'
            wf_gt_fp = f"{wf_gt_folder}/flights/workflows/flights_p{query_id}.json"
            wf_pred_fp = f"{wf_pred_folder}/{model}_flights_test_p{query_id}.json"
        elif query_id >= 92 and query_id <=110:
            target_path = f'{wf_gt_folder}/dish_datasets/cleaned_tables/dish_sample_p{query_id}.csv'
            wf_gt_fp = f"{wf_gt_folder}/dish_datasets/workflows/dish_sample_p{query_id}.json"
            wf_pred_fp = f"{wf_pred_folder}/{model}_dish_test_p{query_id}.json"
        elif query_id >= 62 and query_id <= 91:
            target_path = f'{wf_gt_folder}/ppp_datasets/cleaned_tables/ppp_sample_p{query_id}.csv'
            wf_gt_fp = f"{wf_gt_folder}/ppp_datasets/workflows/ppp_sample_p{query_id}.json"
            wf_pred_fp = f"{wf_pred_folder}/{model}_ppp_test_p{query_id}.json"
        elif query_id >= 31 and query_id <= 61:
            target_path = f'{wf_gt_folder}/CFI_datasets/cleaned_tables/chi_sample_p{query_id}.csv'
            wf_gt_fp = f"{wf_gt_folder}/CFI_datasets/workflows/chi_sample_p{query_id}.json"
            wf_pred_fp = f"{wf_pred_folder}/{model}_chi_test_p{query_id}.json"

        elif query_id <31:
            target_path = f'{wf_gt_folder}/menu_datasets/clean_tables/menu_sample_p{query_id}'
            wf_gt_fp = f"{wf_gt_folder}/menu_datasets/workflows/menu_sample_p{query_id}.json"
            wf_pred_fp = f"{wf_pred_folder}/{model}_menu_test_p{query_id}.json"
        if wf_gt_fp and wf_pred_fp:
            ops_list.append(eval_workflows(query_id, wf_gt_fp, wf_pred_fp))
    ops_result[model] = pd.DataFrame(ops_list)
# ops_df = pd.DataFrame(ops_list)

In [14]:
def eval_workflows(pp_id, gt_wf_fp, pred_wf_fp):
    # print(gt_wf_fp)
    gt_ops_list = parse_recipe(pp_id, recipe=gt_wf_fp)
    pred_ops_list = parse_recipe(pp_id, recipe=pred_wf_fp)
    return {'pp_id': pp_id, 'gt_ops': gt_ops_list[pp_id], 'pred_ops': pred_ops_list[pp_id]}
    # print(gt_ops_list)
    # print(pred_ops_list)

answer_gt_path = 'evaluation/answer_1-154_gt.json'
answer_preds_llama = 'evaluation/answer_1-154_llama3.1.json'


# eval_table_results = eval_answers(answer_gt_path, answer_preds_llama)
# model = models[0]
wf_gt_folder = 'datasets'

query_contents = pd.read_csv('purposes/all_purposes.csv')
    
ops_result = {}
for model in models[:]:
    ops_list = []

    wf_pred_folder = f'CoT.response/{model}/recipes_llm'
    for query_id in range(155):
        row = query_contents[query_contents['ID'] == query_id]
        if len(row) == 0:
            continue
        # if model == 'llama3.1':
        if query_id >126:
            #TODO: what's the point to have the target_path here? 
            target_path = f'{wf_gt_folder}/hospital/clean_tables/hos_pp{query_id}.csv'
            wf_gt_fp = f"{wf_gt_folder}/hospital/workflows/hos_p{query_id}.json"
            wf_pred_fp = f"{wf_pred_folder}/{model}_hos_test_p{query_id}.json"
        elif query_id >= 111 and query_id <=126:
            target_path = f'{wf_gt_folder}/flights/cleaned_tables/flights_data_p{query_id}.csv'
            wf_gt_fp = f"{wf_gt_folder}/flights/workflows/flights_p{query_id}.json"
            wf_pred_fp = f"{wf_pred_folder}/{model}_flights_test_p{query_id}.json"
        elif query_id >= 92 and query_id <=110:
            target_path = f'{wf_gt_folder}/dish_datasets/cleaned_tables/dish_sample_p{query_id}.csv'
            wf_gt_fp = f"{wf_gt_folder}/dish_datasets/workflows/dish_sample_p{query_id}.json"
            wf_pred_fp = f"{wf_pred_folder}/{model}_dish_test_p{query_id}.json"
        elif query_id >= 62 and query_id <= 91:
            target_path = f'{wf_gt_folder}/ppp_datasets/cleaned_tables/ppp_sample_p{query_id}.csv'
            wf_gt_fp = f"{wf_gt_folder}/ppp_datasets/workflows/ppp_sample_p{query_id}.json"
            wf_pred_fp = f"{wf_pred_folder}/{model}_ppp_test_p{query_id}.json"
        elif query_id >= 31 and query_id <= 61:
            target_path = f'{wf_gt_folder}/CFI_datasets/cleaned_tables/chi_sample_p{query_id}.csv'
            wf_gt_fp = f"{wf_gt_folder}/CFI_datasets/workflows/chi_sample_p{query_id}.json"
            wf_pred_fp = f"{wf_pred_folder}/{model}_chi_test_p{query_id}.json"

        elif query_id <31:
            target_path = f'{wf_gt_folder}/menu_datasets/clean_tables/menu_sample_p{query_id}'
            wf_gt_fp = f"{wf_gt_folder}/menu_datasets/workflows/menu_sample_p{query_id}.json"
            wf_pred_fp = f"{wf_pred_folder}/{model}_menu_test_p{query_id}.json"
        if wf_gt_fp and wf_pred_fp:
            ops_list.append(eval_workflows(query_id, wf_gt_fp, wf_pred_fp))
    ops_result[model] = pd.DataFrame(ops_list)
# ops_df = pd.DataFrame(ops_list)

In [15]:
ops_result['llama3.1']

,pp_id,gt_ops,pred_ops
0,1,"[numeric, row_reorder]","[upper, mass_edit, upper, numeric]"
1,2,[numeric],[numeric]
2,3,"[trim, text_transform, mass_edit, mass_edit, m...","[upper, upper]"
3,4,"[trim, mass_edit, mass_edit, mass_edit, upper]","[upper, upper, upper, upper]"
4,5,"[trim, upper, mass_edit, mass_edit, mass_edit]","[upper, mass_edit, upper, upper, upper, upper,..."
...,...,...,...
137,150,"[trim, regexr_transform, upper, mass_edit, mas...","[upper, mass_edit, upper, upper, mass_edit]"
138,151,"[regexr_transform, trim, upper, mass_edit, reg...","[upper, mass_edit, upper, upper, mass_edit]"
139,152,"[trim, upper, mass_edit, regexr_transform, mas...","[upper, upper, mass_edit, mass_edit, mass_edit..."
140,153,"[numeric, upper, mass_edit, mass_edit, trim]","[upper, numeric]"


In [16]:
for key, value in ops_result.items():
    model = key
    ops_df = value
    ops_df['gt_ops_length'] = ops_df['gt_ops'].apply(len)
    ops_df['pred_ops_length'] = ops_df['pred_ops'].apply(len)
    ops_df['gt_ops_set_length'] = ops_df['gt_ops'].apply(lambda x: len(set(x)))
    ops_df['pred_ops_set_length'] = ops_df['pred_ops'].apply(lambda x: len(set(x)))

    ops_length_desc = ops_df.describe().loc['mean']
    workflow_results = calculate_operation_metrics(ops_df['gt_ops'], ops_df['pred_ops'])
    workflow_results['pp_id'] = ops_df['pp_id']
    # print(workflow_results[workflow_results['pp_id'] == 127])
    workflow_results.set_index('pp_id', inplace=True)
    workflow_results.to_csv(f'evaluation/{model}_workflow_results.csv')

In [17]:
def eval_workflows(pp_id, gt_wf_fp, pred_wf_fp):
    # print(gt_wf_fp)
    gt_ops_list = parse_recipe(pp_id, recipe=gt_wf_fp)
    pred_ops_list = parse_recipe(pp_id, recipe=pred_wf_fp)
    return {'pp_id': pp_id, 'gt_ops': gt_ops_list[pp_id], 'pred_ops': pred_ops_list[pp_id]}
    # print(gt_ops_list)
    # print(pred_ops_list)

answer_gt_path = 'evaluation/answer_1-154_gt.json'
answer_preds_llama = 'evaluation/answer_1-154_llama3.1.json'


# eval_table_results = eval_answers(answer_gt_path, answer_preds_llama)
# model = models[0]
wf_gt_folder = 'datasets'

query_contents = pd.read_csv('purposes/all_purposes.csv')
    
ops_result = {}
for model in models[:]:
    ops_list = []

    wf_pred_folder = f'ablation/{model}/recipes_llm'
    for query_id in range(155):
        row = query_contents[query_contents['ID'] == query_id]
        if len(row) == 0:
            continue
        # if model == 'llama3.1':
        if query_id >126:
            #TODO: what's the point to have the target_path here? 
            target_path = f'{wf_gt_folder}/hospital/clean_tables/hos_pp{query_id}.csv'
            wf_gt_fp = f"{wf_gt_folder}/hospital/workflows/hos_p{query_id}.json"
            wf_pred_fp = f"{wf_pred_folder}/base_{model}_hos_test_p{query_id}.json"
        elif query_id >= 111 and query_id <=126:
            target_path = f'{wf_gt_folder}/flights/cleaned_tables/flights_data_p{query_id}.csv'
            wf_gt_fp = f"{wf_gt_folder}/flights/workflows/flights_p{query_id}.json"
            wf_pred_fp = f"{wf_pred_folder}/base_{model}_flights_test_p{query_id}.json"
        elif query_id >= 92 and query_id <=110:
            target_path = f'{wf_gt_folder}/dish_datasets/cleaned_tables/dish_sample_p{query_id}.csv'
            wf_gt_fp = f"{wf_gt_folder}/dish_datasets/workflows/dish_sample_p{query_id}.json"
            wf_pred_fp = f"{wf_pred_folder}/base_{model}_dish_test_p{query_id}.json"
        elif query_id >= 62 and query_id <= 91:
            target_path = f'{wf_gt_folder}/ppp_datasets/cleaned_tables/ppp_sample_p{query_id}.csv'
            wf_gt_fp = f"{wf_gt_folder}/ppp_datasets/workflows/ppp_sample_p{query_id}.json"
            wf_pred_fp = f"{wf_pred_folder}/base_{model}_ppp_test_p{query_id}.json"
        elif query_id >= 31 and query_id <= 61:
            target_path = f'{wf_gt_folder}/CFI_datasets/cleaned_tables/chi_sample_p{query_id}.csv'
            wf_gt_fp = f"{wf_gt_folder}/CFI_datasets/workflows/chi_sample_p{query_id}.json"
            wf_pred_fp = f"{wf_pred_folder}/base_{model}_chi_test_p{query_id}.json"

        elif query_id <31:
            target_path = f'{wf_gt_folder}/menu_datasets/clean_tables/menu_sample_p{query_id}'
            wf_gt_fp = f"{wf_gt_folder}/menu_datasets/workflows/menu_sample_p{query_id}.json"
            wf_pred_fp = f"{wf_pred_folder}/base_{model}_menu_test_p{query_id}.json"
        if wf_gt_fp and wf_pred_fp:
            ops_list.append(eval_workflows(query_id, wf_gt_fp, wf_pred_fp))
    ops_result[model] = pd.DataFrame(ops_list)
# ops_df = pd.DataFrame(ops_list)

Error parsing recipe for 13: [Errno 2] No such file or directory: 'ablation/llama3.1/recipes_llm/base_llama3.1_menu_test_p13.json'
Error parsing recipe for 28: [Errno 2] No such file or directory: 'ablation/llama3.1/recipes_llm/base_llama3.1_menu_test_p28.json'
Error parsing recipe for 31: [Errno 2] No such file or directory: 'ablation/llama3.1/recipes_llm/base_llama3.1_chi_test_p31.json'
Error parsing recipe for 38: [Errno 2] No such file or directory: 'ablation/llama3.1/recipes_llm/base_llama3.1_chi_test_p38.json'
Error parsing recipe for 42: [Errno 2] No such file or directory: 'ablation/llama3.1/recipes_llm/base_llama3.1_chi_test_p42.json'
Error parsing recipe for 52: [Errno 2] No such file or directory: 'ablation/llama3.1/recipes_llm/base_llama3.1_chi_test_p52.json'
Error parsing recipe for 67: [Errno 2] No such file or directory: 'ablation/llama3.1/recipes_llm/base_llama3.1_ppp_test_p67.json'
Error parsing recipe for 68: [Errno 2] No such file or directory: 'ablation/llama3.1/rec

In [18]:
for key, value in ops_result.items():
    model = key
    ops_df = value
    ops_df['gt_ops_length'] = ops_df['gt_ops'].apply(len)
    ops_df['pred_ops_length'] = ops_df['pred_ops'].apply(len)
    ops_df['gt_ops_set_length'] = ops_df['gt_ops'].apply(lambda x: len(set(x)))
    ops_df['pred_ops_set_length'] = ops_df['pred_ops'].apply(lambda x: len(set(x)))

    ops_length_desc = ops_df.describe().loc['mean']
    workflow_results = calculate_operation_metrics(ops_df['gt_ops'], ops_df['pred_ops'])
    workflow_results['pp_id'] = ops_df['pp_id']
    # print(workflow_results[workflow_results['pp_id'] == 127])
    workflow_results.set_index('pp_id', inplace=True)
    workflow_results.to_csv(f'evaluation/base_{model}_workflow_results.csv')

## workflow ttest

In [23]:
dp_gemma2_workflow_result = pd.read_csv('evaluation/base_gemma2_workflow_results.csv')
# dp_gemma2_workflow_result = dp_gemma2_workflow_result.iloc[:,1:]
gemma2_workflow_result = pd.read_csv('evaluation/gemma2_workflow_results.csv')
# gemma2_workflow_result = gemma2_workflow_result.iloc[:, 1:]

dp_llama_workflow_result = pd.read_csv('evaluation/base_llama3.1_workflow_results.csv')
# dp_llama_workflow_result = dp_llama_workflow_result.iloc[:,1:]
llama_workflow_result = pd.read_csv('evaluation/llama3.1_workflow_results.csv')
# llama_workflow_result = llama_workflow_result.iloc[:, 1:]

dp_mistral_workflow_result = pd.read_csv('evaluation/base_mistral_workflow_results.csv')
# dp_mistral_workflow_result = dp_mistral_workflow_result.iloc[:,1:]
mistral_workflow_result = pd.read_csv('evaluation/mistral_workflow_results.csv')
# mistral_workflow_result = mistral_workflow_result.iloc[:, 1:]

dp_gemma2base_workflow_result = pd.read_csv('evaluation/base_gemma2base_workflow_results.csv')
# dp_gemma2base_workflow_result = dp_gemma2base_workflow_result.iloc[:,1:]
gemma2base_workflow_result = pd.read_csv('evaluation/gemma2base_workflow_results.csv')
# gemma2base_workflow_result = gemma2base_workflow_result.iloc[:, 1:]

In [24]:
llama_workflow_result = dp_llama_workflow_result.merge(llama_workflow_result, on='pp_id', how='left', suffixes=('_dp', '_llama3.1'))
gemma2_workflow_result = dp_gemma2_workflow_result.merge(gemma2_workflow_result, on='pp_id', how='left', suffixes=('_dp', '_gemma2'))
mistral_workflow_result = dp_mistral_workflow_result.merge(mistral_workflow_result, on='pp_id', how='left', suffixes=('_dp', '_mistral'))
gemma2base_workflow_result = dp_gemma2base_workflow_result.merge(gemma2base_workflow_result, on='pp_id', how='left', suffixes=('_dp', '_gemma2base'))

In [26]:
llama_workflow_result.columns

Index(['pp_id', 'accuracy_dp', 'precision_dp', 'recall_dp', 'f1_dp',
       'accuracy_llama3.1', 'precision_llama3.1', 'recall_llama3.1',
       'f1_llama3.1'],
      dtype='object')

In [29]:
#TOBEDONE
metric_names = ['accuracy','precision', 'recall', 'f1']
ttest_result = []
models = ['llama3.1', 'gemma2', 'mistral', 'gemma2base']
for m in metric_names:
    model_list = models[:4] + ['dp'] #['llama', 'gemma2', 'mistral', 'gemma2base','dp']
    llama_result = stats.ttest_rel(llama_workflow_result[f'{m}_dp'].values, llama_workflow_result[f'{m}_llama3.1'].values)
    gemma_result = stats.ttest_rel(gemma2_workflow_result[f'{m}_dp'].values, gemma2_workflow_result[f'{m}_gemma2'].values)
    mistral_result = stats.ttest_ind(mistral_workflow_result[f'{m}_dp'].values, mistral_workflow_result[f'{m}_mistral'].values)
    gemma2base_result = stats.ttest_rel(gemma2base_workflow_result[f'{m}_dp'].values, gemma2base_workflow_result[f'{m}_gemma2base'].values)
    ttest_result.append({f'{m}':[llama_result.pvalue.item(), gemma_result.pvalue.item(), mistral_result.pvalue.item(), gemma2base_result.pvalue.item()]})
    # print(gemma_result)
    # print(mistral_result)
    # break

In [30]:
combined_data = {k: v for d in ttest_result for k, v in d.items()}

ttest_result = pd.DataFrame(combined_data)
ttest_result['model'] = ['llama3.1', 'gemma2', 'mistral', 'gemma2base']
ttest_result

,accuracy,precision,recall,f1,model
0,0.481442,2.060067e-18,9.467631e-21,5.566009e-22,llama3.1
1,0.012044,9.182851e-18,1.969022e-33,9.606975e-31,gemma2
2,0.007269,5.813728e-41,1.039484e-39,1.785764e-45,mistral
3,0.000240,2.808258e-29,6.325248e-40,1.303055e-41,gemma2base


## Total workflow performance

In [93]:
def parse_mean(df, col_name):
    total_wf = df.describe().loc['mean']
    # total_wf.columns = [col_name] #rename(columns={"mean": col_name})
    return total_wf

In [ ]:
ops_single = list(ops_result.values())[0]
ops_single.columns


In [10]:
ops_single = pd.read_csv('/projects/bces/lanl2/LLM4DC/dataset-all - all_purposes.csv')

In [ ]:
for i, x in zip(ops_single['pp_id'], ops_single['gt_ops']):
    if 'column_split' in x:
        print(i, x)

In [ ]:
ops_single

In [43]:
from collections import Counter

In [15]:
def parse_ops_list(ops_df, dataset_name):
    flights_ops_list = []
    for op in ops_df:
        flights_ops_list += op
    flights_ops_result = dict(Counter(flights_ops_list))
    for key in ["upper", "trim", "mass_edit", "regexr_transform", "numeric", "date"]:
        if key not in flights_ops_result.keys():
            flights_ops_result[key] = 0
    flights_ops_result['dataset'] = dataset_name
    return flights_ops_result


In [ ]:
ops_df

In [ ]:
ops_single.set_index('ID')

In [53]:
def parse_ops(ops):
    if type(ops) == str:
        ops = ops.split(',')
        ops = [op.strip() for op in ops]
    else:
        ops = [op.strip() for op in ops]
    return ops

In [ ]:
ops_df.loc[111:127]['Operations']

In [65]:
# gt workflow data operations
# ops_df = list(ops_result.values())[0]
ops_df = ops_single
ops_df.set_index('ID', inplace=True)

ops_df['Operations'] = ops_df['Operations'].apply(parse_ops)
result = [] 
# hos_results = workflow_results[workflow_results['pp_id'] >= 127]
hos_ops = ops_df.loc[127:155]['Operations']
result.append(parse_ops_list(hos_ops, 'hospital'))


flights_ops = ops_df.loc[111:127]['Operations']
result.append(parse_ops_list(flights_ops, 'flight'))


ppp_ops = ops_df.loc[62:91]['Operations']
result.append(parse_ops_list(ppp_ops, 'ppp'))

dish_ops = ops_df.loc[92:111]['Operations']
result.append(parse_ops_list(dish_ops, 'dish'))

menu_ops = ops_df.loc[:31]['Operations']
result.append(parse_ops_list(menu_ops, 'menu'))

chi_ops = ops_df.loc[31:62]['Operations']
result.append(parse_ops_list(chi_ops, 'chicago'))


In [66]:
result = pd.DataFrame(result)

In [ ]:
result

In [ ]:
wf_length = []
total_wf = []
print(len(ops_result))
ppp_wf, dish_wf, menu_wf,chi_wf,hos_wf, flights_wf = [],[],[],[],[],[]
col_name = []
output_wf =  []
for key, value in ops_result.items():
    model = key
    ops_df = value
    ops_df['gt_ops_length'] = ops_df['gt_ops'].apply(len)
    ops_df['pred_ops_length'] = ops_df['pred_ops'].apply(len)
    ops_df['gt_ops_set_length'] = ops_df['gt_ops'].apply(lambda x: len(set(x)))
    ops_df['pred_ops_set_length'] = ops_df['pred_ops'].apply(lambda x: len(set(x)))

    ops_length_desc = ops_df.describe().loc['mean']
    ops_length_desc.columns = [model]
    wf_length.append(ops_length_desc)


    workflow_results = calculate_operation_metrics(ops_df['gt_ops'], ops_df['pred_ops'])
    workflow_results['pp_id'] = ops_df['pp_id']
    # print(workflow_results[workflow_results['pp_id'] == 127])
    workflow_results.set_index('pp_id', inplace=True)
    ops_df.set_index('pp_id', inplace=True)
    total_wf.append(parse_mean(workflow_results, f'total__{model}'))
    col_name.append(f'total__{model}')
    
    
    hos_results = workflow_results.loc[127:155]
       
    # hos_results = workflow_results[workflow_results['pp_id'] >= 127]
    total_wf.append(parse_mean(hos_results, f'hos__{model}'))
    hos_ops = ops_df.loc[127:155]
    wf_length.append(parse_mean(hos_ops, f'hos__{model}'))
    col_name.append(f'hos__{model}')


    # flights_results = workflow_results[workflow_results['pp_id'] >= 111][workflow_results['pp_id'] <=126]
    flights_results = workflow_results.loc[111:127]
    flights_ops = ops_df.loc[111:127]
    total_wf.append(parse_mean(flights_results, f'flights__{model}'))
    wf_length.append(parse_mean(flights_ops, f'flights_{model}'))
    col_name.append(f'flights__{model}')

    # ppp_results = workflow_results[workflow_results['pp_id'] >= 62][workflow_results['pp_id'] <=91]
    ppp_results = workflow_results.loc[62:91]
    ppp_ops = ops_df.loc[62:91] 
    wf_length.append(parse_mean(ppp_ops, f'ppp_{model}'))
    total_wf.append(parse_mean(ppp_results, f'ppp__{model}'))
    col_name.append(f'ppp__{model}')

    # dish_results = workflow_results[workflow_results['pp_id'] >= 92][workflow_results['pp_id'] <= 110]
    dish_results = workflow_results.loc[92:111]
    dish_ops = ops_df.loc[92:111] 
    wf_length.append(parse_mean(dish_ops, f'dish_{model}'))
    total_wf.append(parse_mean(dish_results, f'dish__{model}'))
    col_name.append(f'dish__{model}')

    # menu_results = workflow_results[workflow_results['pp_id'] < 31]
    menu_results = workflow_results.loc[:31]
    menu_ops = ops_df.loc[:31] 
    wf_length.append(parse_mean(menu_ops, f'menu_{model}'))
    total_wf.append(parse_mean(menu_results, f'menu__{model}'))
    col_name.append(f'menu__{model}')
    # chi_results = workflow_results[workflow_results['pp_id'] >= 31][workflow_results['pp_id'] <=61]
    chi_results = workflow_results.loc[31:62]
    chi_ops = ops_df.loc[31:62] 
    wf_length.append(parse_mean(chi_ops, f'chi_{model}'))
    total_wf.append(parse_mean(chi_results, f'chi__{model}'))
    col_name.append(f'chi__{model}')
wf_perf = pd.concat(total_wf, axis=1)
wf_perf.columns = col_name
# print(wf_perf)
wf_perf = wf_perf.transpose()
wf_length = pd.concat(wf_length, axis=1)
wf_length.columns = col_name
wf_length = wf_length.transpose()
wf_perf = wf_perf.reset_index()

In [ ]:
wf_length

In [97]:
wf_length.to_csv('evaluation/workflow_length.csv')

In [ ]:
wf_perf

In [99]:
wf_perf.to_csv('workflow_results_llama_mistral_gemma2_base.csv')

# Dataset Eval

In [27]:
# def retrieve_tg_cols(tg_cols_fp="target_columns_list.csv"):
#     id_tg_cols = {}
#     tg_df = pd.read_csv(tg_cols_fp)
#     result_dict = tg_df.set_index('ID')['tg_columns'].to_dict()
#     return result_dict

In [ ]:
result_dict = retrieve_tg_cols("/projects/bces/lanl2/LLM4DC/evaluation/target_column_list.csv")
print(result_dict)
# model = "llama3.1"
# model = "mistral"
# model = "gemma2"
# model = "dirty"
data_gt_folder = "/projects/bces/lanl2/LLM4DC/datasets"

for model in models[:] + ["dirty"]:
    ratio_list = []
    for query_id in range(155):
        # print(query_id)
        tg_cols = result_dict.get(query_id)
        if tg_cols:
            if model=="dirty":
                if query_id >126:
                    target_path = f'{data_gt_folder}/hospital/clean_tables/hos_pp{query_id}.csv'
                    table_preds_path = f'/projects/bces/lanl2/LLM4DC/datasets/hospital/hos_data_p{query_id}.csv'
                elif query_id >= 111 and query_id <=126:
                    target_path = f'{data_gt_folder}/flights/cleaned_tables/flights_data_p{query_id}.csv'
                    table_preds_path = f'/projects/bces/lanl2/LLM4DC/datasets/flights/flights_data_p{query_id}.csv'
                    # target_path = None
                    # table_preds_path = None
                elif query_id >= 92 and query_id <=110:
                    target_path = f'{data_gt_folder}/dish_datasets/cleaned_tables/dish_sample_p{query_id}.csv'
                    table_preds_path = f'/projects/bces/lanl2/LLM4DC/datasets/dish_datasets/dish_data_p{query_id}.csv'
                elif query_id >= 62 and query_id <= 91:
                    target_path = f'{data_gt_folder}/ppp_datasets/cleaned_tables/ppp_sample_p{query_id}.csv'
                    table_preds_path =  f'/projects/bces/lanl2/LLM4DC/datasets/ppp_datasets/ppp_data_p{query_id}.csv'
                elif query_id >= 31 and query_id <= 61:
                    target_path = f'{data_gt_folder}/CFI_datasets/cleaned_tables/chi_sample_p{query_id}.csv'
                    table_preds_path = f'/projects/bces/lanl2/LLM4DC/datasets/CFI_datasets/chi_food_data_p{query_id}.csv'
                elif query_id <31:
                    target_path = f'{data_gt_folder}/menu_datasets/clean_tables/menu_sample_p{query_id}.csv'
                    table_preds_path = f'/projects/bces/lanl2/LLM4DC/datasets/menu_datasets/menu_p{query_id}.csv'
            else:
                llm_folder = f"CoT.response/{model}/datasets_llm"
                data_gt_folder = "/projects/bces/lanl2/LLM4DC/datasets"
                pred_fp = f'/projects/bces/lanl2/LLM4DC/{llm_folder}'
                if query_id >126:
                    target_path = f'{data_gt_folder}/hospital/clean_tables/hos_pp{query_id}.csv'
                    table_preds_path = f'{pred_fp}/{model}_hos_test_p{query_id}.csv'
                elif query_id >= 111 and query_id <=126:
                    target_path = f'{data_gt_folder}/flights/cleaned_tables/flights_data_p{query_id}.csv'
                    table_preds_path = f'{pred_fp}/{model}_flights_test_p{query_id}.csv'
                elif query_id >= 92 and query_id <=110:
                    target_path = f'{data_gt_folder}/dish_datasets/cleaned_tables/dish_sample_p{query_id}.csv'
                    table_preds_path = f'{pred_fp}/{model}_dish_test_p{query_id}.csv'
                elif query_id >= 62 and query_id <= 91:
                    target_path = f'{data_gt_folder}/ppp_datasets/cleaned_tables/ppp_sample_p{query_id}.csv'
                    table_preds_path = f'{pred_fp}/{model}_ppp_test_p{query_id}.csv'
                elif query_id >= 31 and query_id <= 61:
                    target_path = f'{data_gt_folder}/CFI_datasets/cleaned_tables/chi_sample_p{query_id}.csv'
                    table_preds_path = f'{pred_fp}/{model}_chi_test_p{query_id}.csv'
                elif query_id <31:
                    target_path = f'{data_gt_folder}/menu_datasets/clean_tables/menu_sample_p{query_id}.csv'
                    table_preds_path = f'{pred_fp}/{model}_menu_test_p{query_id}.csv'
            if target_path and table_preds_path:
                gt_df = pd.read_csv(target_path)
                preds_df = pd.read_csv(table_preds_path)
                # print(gt_df.head(5), preds_df.head(5))
                res = average_match_ratio(gt_df, preds_df, tg_cols)
                ratio_list.append({'pp_id': query_id, 'ratio': res})
    dt_result = pd.DataFrame(ratio_list)
    dt_result.to_csv(f'evaluation/{model}_table_result.csv')

In [ ]:
dt_result

In [ ]:
total_tab = []
col_name = []
for model in models[:] + ["dirty"]:
    print(model)
    
    table_results = pd.read_csv(f'evaluation/{model}_table_result.csv')
    table_results.set_index('pp_id', inplace=True)

    total_tab.append(parse_mean(table_results, f'total__{model}'))
    col_name.append(f'total__{model}')
    
    
    hos_results = table_results.loc[127:155]
    hos_results.reset_index()
    # hos_results = workflow_results[workflow_results['pp_id'] >= 127]
    total_tab.append(parse_mean(hos_results, f'hos__{model}'))
    
    col_name.append(f'hos__{model}')


    # flights_results = workflow_results[workflow_results['pp_id'] >= 111][workflow_results['pp_id'] <=126]
    flights_results = table_results.loc[111:127]
    flights_results.reset_index()
    total_tab.append(parse_mean(flights_results, f'flights__{model}'))
    col_name.append(f'flights__{model}')

    # ppp_results = workflow_results[workflow_results['pp_id'] >= 62][workflow_results['pp_id'] <=91]
    ppp_results = table_results.loc[62:91]
    ppp_results.reset_index()
    total_tab.append(parse_mean(ppp_results, f'ppp__{model}'))
    col_name.append(f'ppp__{model}')

    # dish_results = workflow_results[workflow_results['pp_id'] >= 92][workflow_results['pp_id'] <= 110]
    dish_results = table_results.loc[92:111]
    dish_results.reset_index()
    total_tab.append(parse_mean(dish_results, f'dish__{model}'))
    col_name.append(f'dish__{model}')

    # menu_results = workflow_results[workflow_results['pp_id'] < 31]
    menu_results = table_results.loc[:31]
    menu_results = menu_results.reset_index()
    total_tab.append(parse_mean(menu_results, f'menu__{model}'))
    col_name.append(f'menu__{model}')
    # chi_results = workflow_results[workflow_results['pp_id'] >= 31][workflow_results['pp_id'] <=61]
    chi_results = table_results.loc[31:62]
    chi_results = chi_results.reset_index()
    total_tab.append(parse_mean(chi_results, f'chi__{model}'))
    col_name.append(f'chi__{model}')
tab_perf = pd.concat(total_tab, axis=1)
tab_perf.columns = col_name
# print(wf_perf)
tab_perf = tab_perf.transpose()
tab_perf.to_csv('table_column_ratio_results_llama_mistral_gemma2.csv')

In [ ]:
dt_result.head()

In [ ]:
stat = dt_result['ratio'].describe()
for idx in ['mean', 'std',	'min',	'max',	'25%',	'50%',	'75%']:
    print(stat.loc[idx])

In [ ]:
hos_dt_results = dt_result[dt_result['pp_id'] >= 127]
stat = hos_dt_results['ratio'].describe()
for idx in ['mean', 'std',	'min',	'max',	'25%',	'50%',	'75%']:
    print(stat.loc[idx])

In [ ]:
flights_dt_results = dt_result[dt_result['pp_id'] >= 111][dt_result['pp_id'] <=126]
stat = flights_dt_results['ratio'].describe()
for idx in ['mean', 'std',	'min',	'max',	'25%',	'50%',	'75%']:
    print(stat.loc[idx])

In [ ]:
ppp_dt_results = dt_result[dt_result['pp_id'] >= 62][dt_result['pp_id'] <=91]
stat = ppp_dt_results['ratio'].describe()
for idx in ['mean', 'std',	'min',	'max',	'25%',	'50%',	'75%']:
    print(stat.loc[idx])

In [ ]:
dish_dt_result =dt_result[dt_result['pp_id'] >= 92][dt_result['pp_id'] <=110]
stat = dish_dt_result['ratio'].describe()
for idx in ['mean', 'std',	'min',	'max',	'25%',	'50%',	'75%']:
    print(stat.loc[idx])

In [ ]:
chi_dt_result =dt_result[dt_result['pp_id'] >= 31][dt_result['pp_id'] <=61]
stat = chi_dt_result['ratio'].describe()
for idx in ['mean', 'std',	'min',	'max',	'25%',	'50%',	'75%']:
    print(stat.loc[idx])

In [ ]:
menu_dt_result =dt_result[dt_result['pp_id'] < 31]

stat = menu_dt_result['ratio'].describe()
for idx in ['mean', 'std',	'min',	'max',	'25%',	'50%',	'75%']:
    print(stat.loc[idx])

## table eval ttest

In [21]:
dirty_table_result = pd.read_csv('/projects/bces/lanl2/LLM4DC/evaluation/dirty_table_result.csv')
dirty_table_result = dirty_table_result.iloc[:,1:]
gemma2_table_result = pd.read_csv('/projects/bces/lanl2/LLM4DC/evaluation/gemma2_table_result.csv')
gemma2_table_result = gemma2_table_result.iloc[:, 1:]
llama_table_result = pd.read_csv('/projects/bces/lanl2/LLM4DC/evaluation/llama3.1_table_result.csv')
llama_table_result = llama_table_result.iloc[:,1:]
mistral_table_result = pd.read_csv('/projects/bces/lanl2/LLM4DC/evaluation/mistral_table_result.csv')
mistral_table_result = mistral_table_result.iloc[:,1:]
gemma2base_table_result = pd.read_csv('/projects/bces/lanl2/LLM4DC/evaluation/gemma2base_table_result.csv')
gemma2base_table_result = gemma2base_table_result.iloc[:, 1:]

In [87]:
dp_gemma2_table_result = pd.read_csv('evaluation/base_gemma2_table_result.csv')
dp_gemma2_table_result = dp_gemma2_table_result.iloc[:,1:]
gemma2_table_result = pd.read_csv('evaluation/gemma2_table_result.csv')
gemma2_table_result = gemma2_table_result.iloc[:, 1:]

dp_llama_table_result = pd.read_csv('evaluation/base_llama3.1_table_result.csv')
dp_llama_table_result = dp_llama_table_result.iloc[:,1:]
llama_table_result = pd.read_csv('evaluation/llama3.1_table_result.csv')
llama_table_result = llama_table_result.iloc[:,1:]

dp_mistral_table_result = pd.read_csv('evaluation/base_mistral_table_result.csv')
dp_mistral_table_result = dp_mistral_table_result.iloc[:,1:]
mistral_table_result = pd.read_csv('evaluation/mistral_table_result.csv')
mistral_table_result = mistral_table_result.iloc[:,1:]

dp_gemma2base_table_result = pd.read_csv('evaluation/base_gemma2base_table_result.csv')
dp_gemma2base_table_result = dp_gemma2base_table_result.iloc[:,1:]
gemma2base_table_result = pd.read_csv('evaluation/gemma2base_table_result.csv')
gemma2base_table_result = gemma2base_table_result.iloc[:, 1:]

In [88]:
llama_table_result = dp_llama_table_result.merge(llama_table_result, on='pp_id', how='left', suffixes=('_dp', '_llama3.1'))
gemma2_table_result = dp_gemma2_table_result.merge(gemma2_table_result, on='pp_id', how='left', suffixes=('_dp', '_gemma2'))
mistral_table_result = dp_mistral_table_result.merge(mistral_table_result, on='pp_id', how='left', suffixes=('_dp', '_mistral'))
gemma2base_table_result = dp_gemma2base_table_result.merge(gemma2base_table_result, on='pp_id', how='left', suffixes=('_dp', '_gemma2base'))


In [89]:
ttest_result = []
for m in ['ratio']:
    model_list = models[:4] + ['dp'] #['llama', 'gemma2', 'mistral', 'dirty']
    llama_result = stats.ttest_rel(llama_table_result[f'{m}_dp'].values, llama_table_result[f'{m}_llama3.1'].values)
    gemma_result = stats.ttest_rel(gemma2_table_result[f'{m}_dp'].values, gemma2_table_result[f'{m}_gemma2'].values)
    mistral_result = stats.ttest_ind(mistral_table_result[f'{m}_dp'].values, mistral_table_result[f'{m}_mistral'].values)
    gemma2base_result = stats.ttest_rel(gemma2base_table_result[f'{m}_dp'].values, gemma2base_table_result[f'{m}_gemma2base'].values)
    ttest_result.append({f'{m}':[llama_result.pvalue.item(), gemma_result.pvalue.item(), mistral_result.pvalue.item(), gemma2base_result.pvalue.item()]})

In [90]:
combined_data = {k: v for d in ttest_result for k, v in d.items()}

ttest_result = pd.DataFrame(combined_data)

In [ ]:
ttest_result['model'] = ['llama3.1', 'gemma2', 'mistral', 'gemma2base']
ttest_result

In [15]:
table_result = llama_table_result.merge(gemma2_table_result, on='pp_id', how='left', suffixes=('_llama3.1', '_gemma2')).merge(mistral_table_result, on='pp_id', how='left', suffixes=('', '_mistral'))

In [16]:
table_result = table_result.merge(dirty_table_result, on='pp_id', how='left', suffixes=('', '_dirty'))

In [ ]:
table_result

In [21]:
ttest_result = []
for m in ['ratio']:
    model_list = models[:3] + ['dirty'] #['llama', 'gemma2', 'mistral', 'dirty']
    llama_result = stats.ttest_rel(table_result[f'{m}_dirty'].values, table_result[f'{m}_llama3.1'].values)
    gemma_result = stats.ttest_rel(table_result[f'{m}_dirty'].values, table_result[f'{m}_gemma2'].values)
    # print(answer_result[f'{m}_llama'].values)
    mistral_result = stats.ttest_ind(table_result[f'{m}_dirty'].values, table_result[f'{m}'].values)
    ttest_result.append({f'{m}':[llama_result.pvalue.item(), gemma_result.pvalue.item(), mistral_result.pvalue.item()]})

In [22]:
combined_data = {k: v for d in ttest_result for k, v in d.items()}

ttest_result = pd.DataFrame(combined_data)

In [ ]:
ttest_result['model'] = ['llama3.1', 'gemma2', 'mistral']
ttest_result

In [29]:
def ttest_table(answer_result, metric_names=['ratio']):
    ttest_result = []
    for m in metric_names:
        model_list = ['llama', 'gemma2', 'mistral', 'dirty']
        llama_result = stats.ttest_rel(answer_result[f'{m}_dirty'].values, answer_result[f'{m}_llama3.1'].values)
        gemma_result = stats.ttest_rel(answer_result[f'{m}_dirty'].values, answer_result[f'{m}_gemma2'].values)
        # print(answer_result[f'{m}_llama'].values)
        mistral_result = stats.ttest_ind(answer_result[f'{m}_dirty'].values, answer_result[f'{m}'].values)
        ttest_result.append({f'{m}':[llama_result.pvalue.item(), gemma_result.pvalue.item(), mistral_result.pvalue.item()]})
    combined_data = {k: v for d in ttest_result for k, v in d.items()}

    ttest_result = pd.DataFrame(combined_data)
    return ttest_result

In [ ]:
## ttest per dataset
ppp_results = table_result[table_result['pp_id'] >= 62][table_result['pp_id'] <=91]
ppp_ttest = ttest_table(ppp_results)
print(ppp_ttest)
dish_result= table_result[table_result['pp_id'] >= 92]
dish_ttest = ttest_table(dish_result)
print(dish_ttest)
menu_results = table_result[table_result['pp_id'] < 31]
menu_ttest = ttest_table(menu_results)
print(menu_ttest)
chi_results = table_result[table_result['pp_id'] >= 31][table_result['pp_id'] <=61]
chi_ttest = ttest_table(chi_results)
print(chi_ttest)

hospital_results = table_result[table_result['pp_id']>=127][table_result['pp_id']<=155]
hospital_ttest = ttest_table(hospital_results)
print(hospital_ttest)
flights_results = table_result[table_result['pp_id']>=111][table_result['pp_id']<=126]
flights_ttest = ttest_table(flights_results)
print(flights_ttest)

# answer eval

In [22]:
par_folder = '/projects/bces/lanl2/LLM4DC'
datafile_path = f'{par_folder}/evaluation/answer_1-154_gt.json'
data = []
with open(datafile_path, 'r') as f:
    for l in f:
        data.append(json.loads(l))

In [ ]:
from bert_score import score

In [24]:
import json
from typing import Union, List, Dict, Any
from difflib import SequenceMatcher
from math import isclose

# Define a utility to convert JSON strings to Python objects
def parse_input(answer: Union[str, float, List, Dict]) -> Any:
    if isinstance(answer, str):
        try:
            # Try to parse JSON strings into Python objects
            return json.loads(answer)
        except json.JSONDecodeError:
            return answer.lower().strip()  # Normalize strings for comparison
    elif isinstance(answer, float):
        return round(answer, 2)  # Round floats to two decimal places if needed
    return answer  # If already in desired format

# Calculate exact match accuracy
def accuracy_metric(gt: Any, pred: Any) -> float:
    return 1.0 if gt == pred else 0.0

# Calculate precision, recall, and F1 for lists (assuming items are unique)
def precision_recall_f1(gt: List, pred: List) -> Dict[str, float]:
    gt_set, pred_set = set(gt), set(pred)
    true_positives = len(gt_set & pred_set)
    precision = true_positives / len(pred_set) if pred_set else 0
    recall = true_positives / len(gt_set) if gt_set else 0
    f1_score = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0
    return {"precision": precision, "recall": recall, "f1": f1_score}

# Calculate semantic distance for string answers using sequence matching
def semantic_similarity(gt: str, pred: str) -> float:
    return SequenceMatcher(None, gt, pred).ratio()  # Returns a ratio between 0 and 1

# Evaluate an answer based on the ground truth
def calculate_answer_metrics(gt: Any, pred: Any) -> Dict[str, float]:
    # Parse inputs
    gt, pred = parse_input(gt), parse_input(pred)
    
    # Initialize results
    results = {"accuracy": 0, "semantic_similarity": 0, "precision": 0, "recall":0, "f1":0}
    
    # Check type and apply appropriate metrics
    if isinstance(gt, float) and isinstance(pred, float):
        results["accuracy"] = 1.0 if isclose(gt, pred, rel_tol=1e-2) else 0.0  # Accuracy for floats with tolerance
        precision, recall = results["accuracy"], results["accuracy"]
        results.update({"precision": precision, "recall": recall, "f1": 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0})
        # results['bertscore'] = score([f"{pred}"], [f"{gt}"], lang='en', verbose=True)
        results["semantic_similarity"] = semantic_similarity(str(gt), str(pred))
    elif isinstance(gt, int) and isinstance(pred, int):
        results["accuracy"] = 1.0 if isclose(gt, pred, rel_tol=1e-2) else 0.0  # Accuracy for floats with tolerance
        precision, recall = results["accuracy"], results["accuracy"]
        results.update({"precision": precision, "recall": recall, "f1": 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0})
        # results['bertscore'] = score([f"{pred}"], [f"{gt}"], lang='en', verbose=True)
        results["semantic_similarity"] = semantic_similarity(str(gt), str(pred))


    elif isinstance(gt, str) and isinstance(pred, str):
        results["accuracy"] = accuracy_metric(gt.lower(), pred.lower())
        precision, recall = results["accuracy"], results["accuracy"]
        results.update({"precision": precision, "recall": recall, "f1": 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0})
        # results['bertscore'] = score([pred], [gt], lang='en', verbose=True)
        results["semantic_similarity"] = semantic_similarity(gt, pred)

    
    elif isinstance(gt, list) and isinstance(pred, list):
        if type(gt[0]) == str:
            gt = [x.lower() for x in gt]
            if len(pred) > 0:
                if type(pred[0]) == str:
                    pred = [x.lower() for x in pred]
        metrics = precision_recall_f1(gt, pred)
        results.update(metrics)
        results["accuracy"] = accuracy_metric(gt, pred)
        # results['bertscore'] = score([f"{pred}"], [f"{gt}"], lang='en', verbose=True)
        similarity = semantic_similarity(f"{gt}", f"{pred}")
        results["semantic_similarity"] = similarity

    
    elif isinstance(gt, dict) and isinstance(pred, dict):
        gt = {key.lower(): value for key, value in gt.items()}
        pred = {key.lower(): value for key, value in pred.items()}
        gt_keys, pred_keys = list(gt.keys()), list(pred.keys())
        if type(gt[gt_keys[0]]) == dict:
            precision_recall_f1_results = []
            for k in gt_keys:
                if k in pred.keys():
                    pred_input = pred[k].values()
                else:
                    pred_input = []
                precision_recall_f1_results.append(precision_recall_f1(gt[k].values(), pred_input))
            precision = sum([x['precision'] for x in precision_recall_f1_results])/len([x['precision'] for x in precision_recall_f1_results])
            recall = sum([x['recall'] for x in precision_recall_f1_results])/len([x['recall'] for x in precision_recall_f1_results])
            f1 =sum([x['f1'] for x in precision_recall_f1_results])/len([x['f1'] for x in precision_recall_f1_results])
            results.update({'precision': precision, 'recall':recall, 'f1': f1})  # Precision/Recall on keys

        elif type(gt[gt_keys[0]]) == list:
            precision_recall_f1_results = []
            for k in gt_keys:
                if k in pred:
                    pred_input = pred[k]
                else: 
                    pred_input = []
                if type(gt[k][0]) ==str:
                    gt_input = [x.lower() for x in gt[k]]
                    if len(pred_input) > 0:
                        if type(pred_input[0]) == str:
                            pred_input = [x.lower() for x in pred_input]    
                else:
                    gt_input = gt[k]
                precision_recall_f1_results.append(precision_recall_f1(gt[k], pred_input)) 
            precision = sum([x['precision'] for x in precision_recall_f1_results])/len([x['precision'] for x in precision_recall_f1_results])
            recall = sum([x['recall'] for x in precision_recall_f1_results])/len([x['recall'] for x in precision_recall_f1_results])
            f1 =sum([x['f1'] for x in precision_recall_f1_results])/len([x['f1'] for x in precision_recall_f1_results])
            results.update({'precision': precision, 'recall':recall, 'f1': f1})  # Precision/Recall on keys
        

        results["accuracy"] = accuracy_metric(gt, pred)
        # Check semantic similarity for each key-value pair
        similarity = [semantic_similarity(str(gt[k]), str(pred.get(k, ""))) for k in gt_keys]
        results["semantic_similarity"] = sum(similarity) / len(similarity) if similarity else 0
        
        # results['bertscore'] = score([f"{pred}"], [f"{gt}"], lang='en', verbose=True)
    p, r, f1 = score([f"{pred}"], [f"{gt}"], lang='en', verbose=True)
    results.update({'bertscore_p': p.detach().cpu().tolist(),
    'bertscore_r': r.detach().cpu().tolist(),
    'bertscore_f1': f1.detach().cpu().tolist()})
    # print(results)


    return results

In [42]:
test_json = "{\"Zip\":{\"0\":96701,\"1\":96704,\"2\":96707,\"3\":96708,\"4\":96749,\"5\":96750,\"6\":96754,\"7\":96791,\"8\":96813,\"9\":96814,\"10\":96815,\"11\":96816,\"12\":96817,\"13\":96821,\"14\":96825,\"15\":96826},\"LoanCount\":{\"0\":1,\"1\":1,\"2\":4,\"3\":1,\"4\":1,\"5\":1,\"6\":1,\"7\":1,\"8\":1,\"9\":1,\"10\":1,\"11\":2,\"12\":1,\"13\":1,\"14\":1,\"15\":1}}"

In [43]:
test_json = parse_input(test_json)

In [ ]:
test_json.keys()
test_json.get('Zip').values()

In [25]:
# LLM-based history update solution
import importlib.util
import inspect
from typing import List
import requests
import json
import re
import difflib
from collections import Counter
# from spellchecker import SpellChecker
from datetime import datetime
import pandas as pd
import ast
import random
import logging 
# from history_update_problem.call_or import export_rows
from call_or import *

# from evaluation.data_compare import calculate_answer_metrics


def load_answer_dataset(datafile_path):
    """
    load json file, each line is a json dictionary

    datafile_path: str
    return: data:  list_of_dictionary
    """
    data = []
    with open(datafile_path, 'r') as f:
        for l in f:
            data.append(json.loads(l))
    return data

def eval_answers(answer_gt_path, answer_preds_llama):
    answer_gt = load_answer_dataset(answer_gt_path)
    answer_gt = pd.DataFrame(answer_gt)
    answer_preds_llama = load_answer_dataset(answer_preds_llama)
    answer_preds_llama = pd.DataFrame(answer_preds_llama)
    answer_compare = answer_gt.merge(answer_preds_llama[['pp_id', 'answer']], on='pp_id', how='left', suffixes=('_gt', '_preds'))

    results = []
    for i, row in answer_compare.iterrows():
        
        gt = row['answer_gt']
        preds = row['answer_preds']
        
        single_result = calculate_answer_metrics(gt, preds)
        
        single_result['pp_id'] = row['pp_id']
        results.append(single_result)
        # break
    return pd.DataFrame(results)

 
    

In [ ]:
# @title single eexample
answer_gt_path = 'evaluation/answer_1-154_gt.json'
answer_gt = load_answer_dataset(answer_gt_path)
answer_gt = pd.DataFrame(answer_gt)
answer_dirty = 'evaluation/answer_1-154_dirty.json'
answer_dirty = load_answer_dataset(answer_dirty)
answer_dirty = pd.DataFrame(answer_dirty)
answer_preds_llama = 'evaluation/answer_1-154_llama3.1.json'
answer_preds_llama = load_answer_dataset(answer_preds_llama)
answer_preds_llama = pd.DataFrame(answer_preds_llama)
answer_compare = answer_gt.merge(answer_preds_llama[['pp_id', 'answer']], on='pp_id', how='left', suffixes=('_gt', '_preds'))
# answer_compare = answer_gt.merge(answer_dirty[['pp_id', 'answer']], on='pp_id', how='left', suffixes=('_gt', '_preds'))

results = []
for i, row in answer_compare.iterrows():
    gt = row['answer_gt']
    preds = row['answer_preds']
    single_result = calculate_answer_metrics(gt, preds)
    single_result['pp_id'] = row['pp_id']
    print(gt, type(gt))
    print(preds, type(preds))
    results.append(single_result)
    break

pd.DataFrame(results)

In [ ]:
# 'dirty', 
models = ['llama3.1',  'mistral', 'gemma2','gemma2base']
for model in models[0:] + ['dirty']: #['mistral', 'gemma2', 'llama3.1']:
    print(model)
    answer_gt_path = '/projects/bces/lanl2/LLM4DC/evaluation/answer_1-154_gt.json'
    # answer_preds_llama = '/projects/bces/lanl2/LLM4DC/evaluation/answer_1-154_llama3.1.json'
    # model = 'gemma2'
    # model = 'dirty'
    # model = 'llama3.1'
    # model = 'mistral'
    answer_preds = f'/projects/bces/lanl2/LLM4DC/evaluation/answer_1-154_{model}.json'
    # answer_preds = f'/projects/bces/lanl2/LLM4DC/evaluation/answer_1-110_dirty.json'
    eval_answer_results = eval_answers(answer_gt_path, answer_preds)
    eval_answer_results['bertscore_p'] = eval_answer_results['bertscore_p'].apply(lambda x: sum(x)/len(x) if len(x) > 0 else None)
    eval_answer_results['bertscore_r'] = eval_answer_results['bertscore_r'].apply(lambda x: sum(x)/len(x) if len(x) > 0 else None)
    eval_answer_results['bertscore_f1'] = eval_answer_results['bertscore_f1'].apply(lambda x: sum(x)/len(x) if len(x) > 0 else None)
    eval_answer_results.to_csv(f'/projects/bces/lanl2/LLM4DC/evaluation/{model}_answer_result.csv')


In [27]:
def calculate_eval_stats(eval_answer_results_text):

    total_stat_table = eval_answer_results_text.describe().loc['mean']
    eval_answer_results_text.set_index('pp_id', inplace=True)

    hos_results = eval_answer_results_text.loc[127:155]
    hos_stat_table = hos_results.describe().loc['mean']
    # print('hos', hos_stat_table)

    flights_results = eval_answer_results_text.loc[111:127]
    flights_stat_table = flights_results.describe().loc['mean']

    ppp_results = eval_answer_results_text.loc[62:92] #[eval_answer_results_text['pp_id'] >= 62][eval_answer_results_text['pp_id'] <=91]
    ppp_stat_table = ppp_results.describe().loc['mean']

    dish_results = eval_answer_results_text.loc[92:111] #[eval_answer_results_text['pp_id'] >= 92]
    dish_stat_table = dish_results.describe().loc['mean']
    
    menu_results = eval_answer_results_text.loc[:31] #[eval_answer_results_text['pp_id'] < 31]
    menu_stat_table = menu_results.describe().loc['mean']

    chi_results = eval_answer_results_text.loc[31:62] #[eval_answer_results_text['pp_id'] >= 31][eval_answer_results_text['pp_id'] <=61]
    chi_stat_table = chi_results.describe().loc['mean']


    final_result = pd.concat([total_stat_table, menu_stat_table, dish_stat_table, chi_stat_table, ppp_stat_table, hos_stat_table, flights_stat_table], axis=1)
    final_result = final_result.transpose()
    final_result['data'] = ['Total', 'Menu', 'Dish', 'CFI','PPP', 'Hospital', 'Flights' ]
    return final_result

In [ ]:
total_stat_df = []
for model in models[:] + ['dirty']: #, 'llama3.1', 'mistral', 'gemma2']:
    eval_answer_results = pd.read_csv(f'/projects/bces/lanl2/LLM4DC/evaluation/{model}_answer_result.csv')
    # evaL_answer_results_flag = eval_answer_results.merge(numeric_pp_tag, on='pp_id', how='left', suffixes=('', '_numeric'))
    # eval_answer_results_text = evaL_answer_results_flag[evaL_answer_results_flag['flag'] == 0]
    final_stat = calculate_eval_stats(eval_answer_results)
    final_stat['model'] = [model] * 7
    total_stat_df.append(final_stat)
total_stat_df = pd.concat(total_stat_df, axis=0)
total_stat_df = total_stat_df.reset_index()
total_stat_df.iloc[:,2:]

In [30]:
total_stat_df.to_csv('answer_performance_table_llama_gemma_mistral.csv', header=True, index=False)

## numeric tags

In [36]:
numeric_pp_tag = pd.read_csv('/projects/bces/lanl2/LLM4DC/evaluation/purposes_category.csv')
numeric_pp_tag.columns = ['pp_id', 'purposes', 'flag']

In [ ]:
evaL_answer_results_flag = eval_answer_results.merge(numeric_pp_tag, on='pp_id', how='left', suffixes=('', '_numeric'))

In [30]:
eval_answer_results_text = evaL_answer_results_flag[evaL_answer_results_flag['flag'] == 0]

In [7]:
def eval_answers_numeric(answer_gt_path, answer_preds_llama):
    answer_gt = load_answer_dataset(answer_gt_path)
    answer_gt = pd.DataFrame(answer_gt)
    answer_preds_llama = load_answer_dataset(answer_preds_llama)
    answer_preds_llama = pd.DataFrame(answer_preds_llama)
    answer_compare = answer_gt.merge(answer_preds_llama[['pp_id', 'answer']], on='pp_id', how='left', suffixes=('_gt', '_preds'))
    answer_compare = answer_compare.merge(numeric_pp_tag, on='pp_id', how='left', suffixes=('', '_numeric'))
    answer_compare = answer_compare[answer_compare['flag'] == 1]
    answer_compare['answer_preds'] = pd.to_numeric(answer_compare['answer_preds'], errors='coerce')
    answer_compare['answer_gt'] = pd.to_numeric(answer_compare['answer_gt'], errors='coerce')
    answer_compare['difference'] = (answer_compare['answer_preds'] - answer_compare['answer_gt']).abs()/answer_compare['answer_gt']
    return answer_compare

In [13]:
total_stat_df = []
for model in ['dirty', 'mistral', 'gemma2', 'llama3.1']:
    answer_gt_path = '/projects/bces/lanl2/LLM4DC/evaluation/answer_1-154_gt.json'
    # answer_preds_llama = '/projects/bces/lanl2/LLM4DC/evaluation/answer_1-110_llama3.1.json'
    # model = 'gemma2'
    # model = 'dirty'
    # model = 'llama3.1'
    # model = 'mistral'
    answer_preds = f'/projects/bces/lanl2/LLM4DC/evaluation/answer_1-154_{model}.json'
    # answer_preds = f'/projects/bces/lanl2/LLM4DC/evaluation/answer_1-110_dirty.json'
    
    eval_answer_results = eval_answers_numeric(answer_gt_path, answer_preds)
    # eval_answer_results.to_csv(f'evaluation/numeric_answer_diff_{model}.csv')
    final_stat = calculate_eval_stats(eval_answer_results[['difference', 'pp_id']])
    final_stat['model'] = [model] * 7
    total_stat_df.append(final_stat)

In [14]:
total_stat_df = pd.concat(total_stat_df, axis=0)

In [15]:
total_stat_df.to_csv('numeric_answer_stats_llamma_mistral_gemma.csv')

## non-numeric answer

In [ ]:
def eval_answers_numeric(answer_gt_path, answer_preds_llama):
    answer_gt = load_answer_dataset(answer_gt_path)
    answer_gt = pd.DataFrame(answer_gt)
    answer_preds_llama = load_answer_dataset(answer_preds_llama)
    answer_preds_llama = pd.DataFrame(answer_preds_llama)
    answer_compare = answer_gt.merge(answer_preds_llama[['pp_id', 'answer']], on='pp_id', how='left', suffixes=('_gt', '_preds'))
    answer_compare = answer_compare.merge(numeric_pp_tag, on='pp_id', how='left', suffixes=('', '_numeric'))
    answer_compare = answer_compare[answer_compare['flag'] == 1]
    answer_compare['answer_preds'] = pd.to_numeric(answer_compare['answer_preds'], errors='coerce')
    answer_compare['answer_gt'] = pd.to_numeric(answer_compare['answer_gt'], errors='coerce')
    answer_compare['difference'] = (answer_compare['answer_preds'] - answer_compare['answer_gt']).abs()/answer_compare['answer_gt']
    return answer_compare

In [46]:
total_stat_df = []
for model in ['dirty', 'mistral', 'gemma2', 'llama3.1']:
    answer_gt_path = '/projects/bces/lanl2/LLM4DC/evaluation/answer_1-154_gt.json'
    # answer_preds_llama = '/projects/bces/lanl2/LLM4DC/evaluation/answer_1-110_llama3.1.json'
    # model = 'gemma2'
    # model = 'dirty'
    # model = 'llama3.1'
    # model = 'mistral'
    answer_preds = f'/projects/bces/lanl2/LLM4DC/evaluation/answer_1-154_{model}.json'
    # answer_preds = f'/projects/bces/lanl2/LLM4DC/evaluation/answer_1-110_dirty.json'
    eval_answer_results = pd.read_csv(f'/projects/bces/lanl2/LLM4DC/evaluation/{model}_answer_result.csv')
    evaL_answer_results_flag = eval_answer_results.merge(numeric_pp_tag, on='pp_id', how='left', suffixes=('', '_numeric'))
    eval_answer_results_text = evaL_answer_results_flag[evaL_answer_results_flag['flag'] == 0]
    # print(eval_answer_results_text)
    final_stat = calculate_eval_stats(eval_answer_results_text)
    final_stat['model'] = [model] * 7
    total_stat_df.append(final_stat)

In [ ]:
total_stat_df = pd.concat(total_stat_df, axis=0)
total_stat_df

In [48]:
total_stat_df.to_csv('non_numeric_answer_results_llama_mistral_gemma.csv')

In [21]:
models = ['llama3.1',  'mistral', 'gemma2','deepseek-r1']


In [37]:
dish_results = eval_answer_results_text[eval_answer_results_text['pp_id'] >= 92]
dish_stat_table = dish_results.describe().loc['mean']

In [ ]:
dish_stat_table

In [ ]:
eval_answer_results.describe()

In [ ]:
ppp_results = eval_answer_results[eval_answer_results['pp_id'] >= 62][eval_answer_results['pp_id'] <=91]

In [38]:
dish_results = eval_answer_results[eval_answer_results['pp_id'] >= 92]

In [ ]:
chi_results = eval_answer_results[eval_answer_results['pp_id'] >= 31][eval_answer_results['pp_id'] <=61]

In [66]:
menu_results = eval_answer_results[eval_answer_results['pp_id'] < 31]

In [ ]:
ppp_results.describe()

In [ ]:
dish_results.describe()

In [ ]:
menu_results.describe()

In [ ]:
chi_results.describe()

In [34]:
eval_answer_results = pd.DataFrame(eval_answer_results)

In [ ]:
eval_answer_results['accuracy'].sum()

In [ ]:
eval_answer_results.describe()

## answer eval ttest


In [28]:
from scipy import stats

In [ ]:
dirty_answer_result = pd.read_csv('/projects/bces/lanl2/LLM4DC/evaluation/dirty_answer_result.csv')
dirty_answer_result = dirty_answer_result.iloc[:,1:]

dp_gemma2_answer_result = pd.read_csv('/projects/bces/lanl2/LLM4DC/evaluation/base_gemma2_answer_result.csv')
dp_gemma2_answer_result = dp_gemma2_answer_result.iloc[:, 1:]
gemma2_answer_result = pd.read_csv('/projects/bces/lanl2/LLM4DC/evaluation/gemma2_answer_result.csv')
gemma2_answer_result = gemma2_answer_result.iloc[:, 1:]

dp_llama_answer_result = pd.read_csv('/projects/bces/lanl2/LLM4DC/evaluation/base_llama3.1_answer_result.csv')
dp_llama_answer_result = dp_llama_answer_result.iloc[:, 1:]
llama_answer_result = pd.read_csv('/projects/bces/lanl2/LLM4DC/evaluation/llama3.1_answer_result.csv')
llama_answer_result = llama_answer_result.iloc[:,1:]

dp_mistral_answer_result = pd.read_csv('/projects/bces/lanl2/LLM4DC/evaluation/base_mistral_answer_result.csv')
dp_mistral_answer_result = dp_mistral_answer_result.iloc[:, 1:]
mistral_answer_result = pd.read_csv('/projects/bces/lanl2/LLM4DC/evaluation/mistral_answer_result.csv')
mistral_answer_result = mistral_answer_result.iloc[:,1:]

dp_gemma2base_answer_result = pd.read_csv('/projects/bces/lanl2/LLM4DC/evaluation/base_gemma2base_answer_result.csv')
dp_gemma2base_answer_result = dp_gemma2base_answer_result.iloc[:, 1:]
gemma2base_answer_result = pd.read_csv('/projects/bces/lanl2/LLM4DC/evaluation/gemma2base_answer_result.csv')
gemma2base_answer_result = gemma2base_answer_result.iloc[:, 1:]


In [75]:
# dirty_answer_result = pd.read_csv('evaluation/dirty_answer_result.csv')
# dirty_answer_result = dirty_answer_result.iloc[:,1:]

dp_gemma2_answer_result = pd.read_csv('evaluation/base_gemma2_answer_result.csv')
dp_gemma2_answer_result = dp_gemma2_answer_result.iloc[:, 1:]
gemma2_answer_result = pd.read_csv('evaluation/gemma2_answer_result.csv')
gemma2_answer_result = gemma2_answer_result.iloc[:, 1:]

dp_llama_answer_result = pd.read_csv('evaluation/base_llama3.1_answer_result.csv')
dp_llama_answer_result = dp_llama_answer_result.iloc[:, 1:]
llama_answer_result = pd.read_csv('evaluation/llama3.1_answer_result.csv')
llama_answer_result = llama_answer_result.iloc[:,1:]

dp_mistral_answer_result = pd.read_csv('evaluation/base_mistral_answer_result.csv')
dp_mistral_answer_result = dp_mistral_answer_result.iloc[:, 1:]
mistral_answer_result = pd.read_csv('evaluation/mistral_answer_result.csv')
mistral_answer_result = mistral_answer_result.iloc[:,1:]

dp_gemma2base_answer_result = pd.read_csv('evaluation/base_gemma2base_answer_result.csv')
dp_gemma2base_answer_result = dp_gemma2base_answer_result.iloc[:, 1:]
gemma2base_answer_result = pd.read_csv('evaluation/gemma2base_answer_result.csv')
gemma2base_answer_result = gemma2base_answer_result.iloc[:, 1:]


In [76]:
llama_answer_result = dp_llama_answer_result.merge(llama_answer_result, on='pp_id', how='left', suffixes=('_dp', '_llama3.1'))
gemma2_answer_result = dp_gemma2_answer_result.merge(gemma2_answer_result, on='pp_id', how='left', suffixes=('_dp', '_gemma2'))
mistral_answer_result = dp_mistral_answer_result.merge(mistral_answer_result, on='pp_id', how='left', suffixes=('_dp', '_mistral'))
gemma2base_answer_result = dp_gemma2base_answer_result.merge(gemma2base_answer_result, on='pp_id', how='left', suffixes=('_dp', '_gemma2base'))

In [ ]:
mistral_answer_result.columns

In [84]:
#TOBEDONE
metric_names = ['accuracy','semantic_similarity', 'precision', 'recall', 'f1', 'bertscore_p', 'bertscore_r','bertscore_f1']
ttest_result = []
models = ['llama3.1', 'gemma2', 'mistral', 'gemma2base']
for m in metric_names:
    model_list = models[:4] + ['dp'] #['llama', 'gemma2', 'mistral', 'gemma2base','dp']
    llama_result = stats.ttest_rel(llama_answer_result[f'{m}_dp'].values, llama_answer_result[f'{m}_llama3.1'].values)
    gemma_result = stats.ttest_rel(gemma2_answer_result[f'{m}_dp'].values, gemma2_answer_result[f'{m}_gemma2'].values)
    mistral_result = stats.ttest_ind(mistral_answer_result[f'{m}_dp'].values, mistral_answer_result[f'{m}_mistral'].values)
    gemma2base_result = stats.ttest_rel(gemma2base_answer_result[f'{m}_dp'].values, gemma2base_answer_result[f'{m}_gemma2base'].values)
    ttest_result.append({f'{m}':[llama_result.pvalue.item(), gemma_result.pvalue.item(), mistral_result.pvalue.item(), gemma2base_result.pvalue.item()]})
    # print(gemma_result)
    # print(mistral_result)
    # break

In [ ]:
combined_data = {k: v for d in ttest_result for k, v in d.items()}

ttest_result = pd.DataFrame(combined_data)

In [ ]:
ttest_result['model'] = ['llama3.1', 'gemma2', 'mistral', 'gemma2base']
ttest_result

In [62]:
answer_result = llama_answer_result.merge(gemma2_answer_result, on='pp_id', how='left', suffixes=('_llama3.1', '_gemma2')).merge(mistral_answer_result, on='pp_id', how='left', suffixes=('', '_mistral'))

In [74]:
answer_result = llama_answer_result.merge(gemma2_answer_result, on='pp_id', how='left', suffixes=('_llama3.1', '_mistral'))

In [ ]:
answer_result = answer_result.merge(gemma2_answer_result, on='pp_id', how='left', suffixes=('', '_gemma2'))

In [ ]:
answer_result

In [55]:
answer_result = answer_result.merge(dirty_answer_result, on='pp_id', how='left', suffixes=('', '_dirty'))

In [ ]:
answer_result.columns

In [26]:
metric_names = ['accuracy','semantic_similarity', 'precision', 'recall', 'f1', 'bertscore_p', 'bertscore_r','bertscore_f1']


In [34]:
ttest_result = []
for m in metric_names:
    model_list = models[:4] + ['dirty'] #['llama', 'gemma2', 'mistral', 'gemma2base','dirty']
    llama_result = stats.ttest_rel(answer_result[f'{m}_dirty'].values, answer_result[f'{m}_llama3.1'].values)
    gemma_result = stats.ttest_rel(answer_result[f'{m}_dirty'].values, answer_result[f'{m}_gemma2'].values)
    # print(answer_result[f'{m}_llama'].values)
    mistral_result = stats.ttest_ind(answer_result[f'{m}_dirty'].values, answer_result[f'{m}'].values)
    gemma2base_result = stats.ttest_rel(answer_result[f'{m}_dirty'].values, answer_result[f'{m}_gemma2base'].values)
    ttest_result.append({f'{m}':[llama_result.pvalue.item(), gemma_result.pvalue.item(), mistral_result.pvalue.item()]})
    # print(gemma_result)
    # print(mistral_result)
    # break

In [35]:
def ttest_answer(answer_result):
    ttest_result = []
    for m in metric_names:
        model_list = ['llama', 'gemma2', 'mistral', 'gemma2base', 'dirty']
        llama_result = stats.ttest_rel(answer_result[f'{m}_dirty'].values, answer_result[f'{m}_llama3.1'].values)
        gemma_result = stats.ttest_rel(answer_result[f'{m}_dirty'].values, answer_result[f'{m}_gemma2'].values)
        # print(answer_result[f'{m}_llama'].values)
        mistral_result = stats.ttest_ind(answer_result[f'{m}_dirty'].values, answer_result[f'{m}'].values)
        gemma2base_result = stats.ttest_rel(answer_result[f'{m}_dirty'].values, answer_result[f'{m}_gemma2base'].values)
        ttest_result.append({f'{m}':[llama_result.pvalue.item(), gemma_result.pvalue.item(), mistral_result.pvalue.item(), gemma2base_result.pvalue.item()]})
    combined_data = {k: v for d in ttest_result for k, v in d.items()}

    ttest_result = pd.DataFrame(combined_data)
    return ttest_result

In [ ]:
combined_data = {k: v for d in ttest_result for k, v in d.items()}

ttest_result = pd.DataFrame(combined_data)


In [ ]:
ttest_result['model'] = ['llama3.1', 'gemma2', 'mistral', 'gemma2base']

In [ ]:
ttest_result

## ttest per dataset

In [ ]:
ppp_results = answer_result[answer_result['pp_id'] >= 62][answer_result['pp_id'] <=91]
ppp_ttest = ttest_answer(ppp_results)

In [ ]:
ppp_ttest

In [ ]:
dish_result= answer_result[answer_result['pp_id'] >= 92]
dish_ttest = ttest_answer(dish_result)
dish_ttest

In [ ]:
menu_results = answer_result[answer_result['pp_id'] < 31]
menu_ttest = ttest_answer(menu_results)
menu_ttest

In [ ]:
chi_results = answer_result[answer_result['pp_id'] >= 31][answer_result['pp_id'] <=61]
chi_ttest = ttest_answer(chi_results)
chi_ttest

In [ ]:
ppp_results = answer_result[answer_result['pp_id'] >= 62][answer_result['pp_id'] <=91]
ppp_ttest = ttest_answer(ppp_results)
ppp_ttest

In [ ]:
hospital_results = answer_result[answer_result['pp_id']>=127][answer_result['pp_id']<=155]
hospital_ttest = ttest_answer(hospital_results)
hospital_ttest

In [ ]:
flights_results = answer_result[answer_result['pp_id']>=111][answer_result['pp_id']<=126]
flights_ttest = ttest_answer(flights_results)
flights_ttest

# Visualize

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
plt.style.available

In [3]:
import pandas as pd
df_viz = pd.read_csv('/projects/bces/lanl2/LLM4DC/workflow_length_for_viz.csv')

In [ ]:
df_viz.columns

In [ ]:
df_viz

In [6]:
predicted_length = df_viz['pred_ops_length']
predicted_length = np.array(predicted_length).reshape(6, 4)

In [7]:
predicted_set_size = np.array(df_viz['pred_ops_set_length']).reshape(6,4)

In [ ]:
df_viz['Dataset'].unique()

In [ ]:


# Use an academic style for publication (ACL)
plt.style.use('seaborn-v0_8-white') #-whitegrid')

# Define datasets and models
datasets = ['PPP', 'CFI', 'Dish', 'Menu', 'Flights', 'Hospital']
models = ['Ground Truth', 'Llama 3.1', 'Mistral', 'Gemma 2']

n_models = len(models)
group_labels = ['Predicted Length', 'Predicted Set Size']
n_groups = len(group_labels)

# Example data arrays: each row corresponds to one dataset,
# and each column corresponds to one model.
# Define a color-blind friendly palette for 4 models
# model_colors = ["#377eb8", "#e41a1c", "#4daf4a", "#984ea3"]
model_colors = ["#3d88e5", "#d82360", "#f8c109", "#09372f"]

# Set up a 2 x 3 grid of subplots with constrained layout for a tighter fit
fig, axes = plt.subplots(2, 3, figsize=(18, 9), constrained_layout=True)
axes = axes.flatten()

# Bar width for each model within a group
bar_width = 0.15
# Total width occupied by all model bars in a group
total_bar_width = n_models * bar_width

# x positions for the groups (center positions)
group_x = np.arange(n_groups)

for idx, ax in enumerate(axes):
    if idx >= len(datasets):
        break

    # For each dataset, extract values for predicted length and set size.
    length_vals = predicted_length[idx]
    set_size_vals = predicted_set_size[idx]
    
    # Plot bars for each group
    for group in range(n_groups):
        # For each model, compute its x position within the group.
        for j in range(n_models):
            # Calculate the x position so that bars are centered in each group.
            x_pos = group_x[group] - (total_bar_width / 2) + j * bar_width + bar_width / 2
            # Choose the appropriate value for the current group
            value = length_vals[j] if group == 0 else set_size_vals[j]
            # Draw the bar
            bar = ax.bar(x_pos, value, width=bar_width, color=model_colors[j],
                         label=models[j] if group == 0 and idx == 0 else "")
            # Add value label above each bar
            ax.text(x_pos, value, f'{round(value,2)}', ha='center', va='bottom', fontsize=12)

    # Set title and labels for the subplot
    ax.set_title(datasets[idx], fontsize=16)
    ax.set_xticks(group_x)
    ax.set_xticklabels(group_labels, fontsize=14)
    # ax.set_xlabel('Metrics', fontsize=14)
    ax.set_ylabel('Count', fontsize=14)

# Only add legend to the first subplot to avoid redundancy
axes[0].legend(fontsize=14, loc='upper right')

plt.savefig("ops_length_viz.pdf", format="pdf")

plt.show()

In [8]:
df_ops_stats_viz=pd.read_csv('/projects/bces/lanl2/LLM4DC/ops_stats_viz.csv')

In [ ]:
df_ops_stats_viz.iloc[:,1:].values

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Use an academic style for publication
plt.style.use('seaborn-v0_8-white')
plt.rcParams.update({'font.size': 14})

# Define operations and datasets
operations = ["upper","trim","numeric","date","regexr_transform","mass_edit"]
datasets = [ 'Menu', 'Dish','PPP', 'Flights', 'Hospital', 'CFI']
n_datasets = len(datasets)
n_operations = len(operations)

# Example count data: each row corresponds to one operation,
# and each column corresponds to one dataset.
# This array should be of shape (6, 6).
count_data = df_ops_stats_viz.iloc[:,1:].values.transpose()

# Define a color-blind friendly palette (ColorBrewer-inspired) for 6 datasets
colors = ["#3d88e5", "#d82360", "#f8c109", "#09372f", "#ff6d01", "#46bdc6"]

# Set up a single subplot with constrained layout
fig, ax = plt.subplots(figsize=(10.5, 5), constrained_layout=True)

# Bar width for each dataset's bar in a group
bar_width = 0.12
total_bar_width = n_datasets * bar_width  # total width for one operation group

# x positions for each operation group (centered)
group_x = np.arange(n_operations)

# Plot bars for each dataset within each operation group
for i in range(n_operations):
    # Compute the x positions for the current dataset's bars in each group.
    x_positions = group_x - (total_bar_width / 2) + i * bar_width + bar_width / 4
    ax.bar(x_positions, count_data[:, i], width=bar_width, color=colors[i], label=operations[i])
    
    # Add value labels above each bar
    for j, value in enumerate(count_data[:, i]):
        ax.text(x_positions[j], value, f'{value}', ha='center', va='bottom', fontsize=12)

# Set x-axis labels and title
ax.set_xticks(group_x)
# ax.set_xticklabels(operations, fontsize=14)
ax.set_xticklabels(datasets, fontsize=14)
# ax.set_xlabel('Operations', fontsize=14)
ax.set_ylabel('Count', fontsize=14)
# ax.set_title('Count of Each Operation by Dataset', fontsize=16)

# Place the legend at the top in one line
ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.1), ncol=n_datasets,columnspacing=0.5, fontsize=14)
# plt.legend(loc='upper center', bbox_to_anchor=(0.5, 1.15, 0.5, 0.2), ncol=2, columnspacing=0.5, handlelength=1, handletextpad=0.2)

# Save the plot as a PDF file
plt.savefig("ops_stats_viz.pdf", format="pdf")
plt.show()